# R3-CAP · Capstone — un asistente de IA para tu servicio — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R3 · *IA Aplicada* · Semana 6–7 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Adaptar **tu propio corpus** (o uno del banco) al formato del asistente.
2. Construir el **índice** con la mejor configuración de R3-03.
3. **Evaluar** con preguntas reales (hit@1, hit@3) antes de confiar.
4. Responder **con citas** y **abstenerse** cuando no hay respuesta.
5. Documentar **riesgos y ética** en una ficha de uso responsable.

**Competencia de salida:** diseñar, evaluar y documentar un asistente de IA generativa para un problema real del servicio, con citas, abstención y supervisión humana.

**Dato real:** las 245 fichas de trámites de ChileAtiende (ejemplo) o los documentos de tu organismo.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, re, json, urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 👉 Cambia esto por tu propio corpus (un CSV con una fila por documento).
CORPUS_CSV = "fichas.csv"
if not os.path.exists(CORPUS_CSV):                     # "en vivo o caché"
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{CORPUS_CSV}", CORPUS_CSV)

crudo = pd.read_csv(CORPUS_CSV)
print("Corpus crudo:", crudo.shape)
crudo.head(3)

### El LLM detrás de una sola función

La misma `preguntar_llm(prompt, respaldo=...)` de toda la rama. Con `GEMINI_API_KEY` en los
Secretos de Colab responde en vivo; sin ella devuelve el respaldo. **Tu capstone debe correr
completo sin clave** (es el criterio de reproducibilidad de la rúbrica).

> Si en tu entorno falta el SDK de Google, instálalo con `!pip install -q google-genai`.

In [ ]:
# ── preguntar_llm(): el LLM (Gemini) detrás de UNA función ─────────────────────
# Cambiar de proveedor o de modelo = cambiar solo esta celda.
# En Colab: 🔑 (Secretos) → agrega GEMINI_API_KEY (gratis en aistudio.google.com).
import os

MODELO = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash-lite")

def _clave_gemini():
    try:                                   # Colab: secreto del notebook
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:                      # local: variable de entorno
        return os.environ.get("GEMINI_API_KEY")

def preguntar_llm(prompt, respaldo="[respuesta de respaldo — configura GEMINI_API_KEY para usar el LLM en vivo]"):
    """Envía el prompt a Gemini y devuelve el texto. Sin clave o sin red, devuelve
    `respaldo` para que el notebook siga corriendo (y se pueda verificar en CI)."""
    clave = _clave_gemini()
    if not clave:
        return respaldo
    try:
        from google import genai
        cliente = genai.Client(api_key=clave)
        return cliente.models.generate_content(model=MODELO, contents=prompt).text
    except Exception as e:
        print("⚠️ No se pudo llamar al LLM:", e)
        return respaldo

def get_llm():
    """Compatibilidad: devuelve una función generate(prompt)->str."""
    return preguntar_llm

print("Modelo:", MODELO, "| modo:", "EN VIVO" if _clave_gemini() else "RESPALDO (sin clave)")

## 0. Enmarca tu proyecto

Antes de escribir código, responde en una frase cada una:

> **¿Quién pregunta?** _(ciudadanía, funcionarios de otra unidad, tu propia jefatura…)_
>
> **¿Qué pregunta?** _(el tipo de consultas que hoy se responden a mano, una y otra vez)_
>
> **¿Con qué documentos o datos se responde?** _(y quién es su dueño)_
>
> **¿Qué pasa si el asistente se equivoca?** _(esto define cuánta supervisión humana necesitas)_

### Elige tu corpus (en este orden de preferencia)

1. **Documentos o datos de tu propio servicio** — el capstone vale el doble si resuelve un
   problema real: preguntas frecuentes, instructivos, reglamentos internos, respuestas tipo de
   la OIRS… **siempre públicos y sin datos personales**.
2. El **banco del bootcamp** (todos reales, en `data/` del repositorio):
   `fichas.csv` (245 trámites de ChileAtiende con texto completo — el ejemplo de esta plantilla) ·
   `matricula.csv` (matrícula escolar 2025, MINEDUC) · `aire_diario.csv` + `estaciones.csv`
   (PM2.5 diario SINCA) · `reclamos.csv` (reclamos de consumo SERNAC, por categoría) ·
   `sismos.csv` (catálogo USGS).
3. Cualquier conjunto de **datos.gob.cl**.

### Dos caminos

| Si tu corpus es… | Construye… | Plantilla |
|---|---|---|
| **Texto** (fichas, instructivos, normas) | Un **asistente RAG** que responde citando y se abstiene | Este notebook |
| **Tablas** (matrícula, aire, tus registros) | Un **agente** con herramientas, traza y aprobación humana | R3-04 (cambia las herramientas por consultas a tus datos) |

> Esta plantilla sigue el camino **RAG** con `fichas.csv`. Para usar tus documentos, ajusta
> `CORPUS_CSV` y el diccionario `COLUMNAS` del ejercicio 1; el resto funciona igual.

## 1. Prepara el corpus (el adaptador)

Cada fuente trae sus columnas con otros nombres. El **adaptador** traduce tu tabla al formato que espera el resto del notebook: `id`, `titulo`, `url` y `texto` (la `url` es la **cita**; si tus documentos no tienen url, usa el nombre del archivo o el número de resolución). Además limpia: sin textos vacíos y sin `id` repetidos.

### ✍️ Ejercicio 1 — Completa `preparar_corpus`

In [ ]:
# Nombre de TU columna para cada campo que necesita el asistente
COLUMNAS = {"id": "id", "titulo": "titulo", "url": "url", "texto": "texto"}

def preparar_corpus(df, columnas=COLUMNAS):
    corpus = df.rename(columns={v: k for k, v in columnas.items()})[list(columnas)]
    corpus = corpus.dropna(subset=["texto"])
    corpus = corpus[corpus["texto"].str.strip() != ""]
    corpus = corpus.drop_duplicates(subset="id")
    return corpus.reset_index(drop=True)

corpus = preparar_corpus(crudo)
print("Documentos:", len(corpus))
corpus.head(3)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert list(corpus.columns) == ["id", "titulo", "url", "texto"]
    assert corpus["id"].is_unique, "no puede haber id repetidos"
    prueba = pd.DataFrame({"id": [1, 1, 2, 3], "titulo": ["a", "a", "b", "c"], "url": ["u", "u", "v", "w"],
                           "texto": ["hola", "hola", None, "  "]})
    assert len(preparar_corpus(prueba)) == 1, "debe quitar duplicados, nulos y textos vacíos"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'corpus = corpus.dropna(subset=["texto"]); corpus = corpus[corpus["texto"].str.strip() != ""]; corpus = corpus.drop_duplicates(subset="id").')
    print("   Detalle:", e)

## 2. Construye el índice

Usamos la receta que mejor funcionó en la profundización de R3-03: el **título repetido 3 veces** y n-gramas de **caracteres** (3 a 5), que absorben variaciones como «pescar»/«pesca». El tamaño de trozo (aquí 120 palabras; en R3-03 también rindió bien 60) depende de tu corpus: pruébalo con tu evaluación del ejercicio 3. `recuperar` ya está escrita; completa `construir_indice`.

### ✍️ Ejercicio 2 — Completa `construir_indice`

In [ ]:
def trocear(texto, max_palabras=120):
    """Corta en cada subtítulo '## ' o al pasar de max_palabras (la función de R3-03)."""
    trozos, actual, n = [], [], 0
    for linea in str(texto).split("\n"):
        if not linea.strip():                    # las líneas en blanco no aportan
            continue
        palabras = len(linea.split())
        if actual and (linea.startswith("## ") or n + palabras > max_palabras):
            trozos.append("\n".join(actual))
            actual, n = [], 0
        actual.append(linea)
        n += palabras
    if actual:
        trozos.append("\n".join(actual))
    return trozos

def construir_indice(corpus, max_palabras=120, peso_titulo=3):
    trozos = pd.DataFrame([{"id": d.id, "titulo": d.titulo, "url": d.url, "texto": t}
                           for d in corpus.itertuples() for t in trocear(d.texto, max_palabras)])
    vect = TfidfVectorizer(strip_accents="unicode", analyzer="char_wb", ngram_range=(3, 5))
    X = vect.fit_transform((trozos["titulo"] + "\n") * peso_titulo + trozos["texto"])
    return {"trozos": trozos, "vect": vect, "X": X}

def recuperar(indice, pregunta, k=3):
    puntajes = (indice["X"] @ indice["vect"].transform([pregunta]).T).toarray().ravel()
    mejores = np.argsort(-puntajes, kind="stable")[:k]
    res = indice["trozos"].iloc[mejores].copy()
    res["puntaje"] = puntajes[mejores]
    return res

indice = construir_indice(corpus)
print("Trozos:", len(indice["trozos"]), "| matriz:", indice["X"].shape)
recuperar(indice, "¿Cómo me afilio a Fonasa?")[["id", "titulo", "puntaje"]]

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert indice["X"].shape[0] == len(indice["trozos"]), "una fila de X por trozo"
    assert set(indice["trozos"]["id"]) == set(corpus["id"]), "todos los documentos deben quedar indexados"
    r = recuperar(indice, "¿Cómo me afilio a Fonasa?")
    assert len(r) == 3 and list(r["puntaje"]) == sorted(r["puntaje"], reverse=True)
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'vect = TfidfVectorizer(strip_accents="unicode", analyzer="char_wb", ngram_range=(3, 5)); X = vect.fit_transform((trozos["titulo"] + "\\n") * peso_titulo + trozos["texto"]).')
    print("   Detalle:", e)

## 3. Evalúa con TUS preguntas

Sin evaluación no hay capstone. Escribe al menos **8 preguntas** reales con el documento que debería aparecer, y mide **hit@1** y **hit@3** (R3-03). Si tu hit@3 queda bajo 0,7, vuelve al ejercicio 2 y prueba otra configuración — y **anótalo** en tu informe.

### ✍️ Ejercicio 3 — Arma tu `informe_eval`

In [ ]:
# Tus preguntas de evaluación: (pregunta, id del documento que DEBERÍA aparecer).
# Escríbelas ANTES de mirar resultados. Estas son de ejemplo para el corpus de fichas:
# reemplázalas por preguntas reales de las personas que atiende tu servicio.
MIS_PREGUNTAS = [
    ("¿Cuándo se paga el aguinaldo de Fiestas Patrias a los pensionados?", 26553),
    ("¿Qué es la Pensión Garantizada Universal y cuánto aumentó?", 130457),
    ("¿Cómo cambio mi domicilio si soy pensionado del IPS y vivo afuera?", 6347),
    ("¿Cómo pido asesoría previsional gratuita?", 9648),
    ("¿Cómo obtengo el certificado de alumno prioritario?", 20613),
    ("¿Qué es el Bolsillo Familiar Electrónico?", 115303),
    ("¿Cómo postulo al Capital Semilla Emprende?", 20608),
    ("¿Cómo pido la Beca Bicentenario?", 2227),
    ("¿Quién recibe el Aporte Familiar Permanente?", 38913),
    ("¿Cómo me inscribo en la Bolsa Nacional de Empleo?", 5072),
    ("Me rechazaron la licencia porque dicen que no hice reposo, ¿qué hago?", 3976),
    ("¿Hay ayuda para pagar la casa de reposo de mi mamá?", 5039),
]

def hit_at_k(indice, preguntas, k):
    return sum(e in set(recuperar(indice, p, k)["id"]) for p, e in preguntas) / len(preguntas)

informe_eval = {"n_preguntas": len(MIS_PREGUNTAS),
                "hit@1": hit_at_k(indice, MIS_PREGUNTAS, 1),
                "hit@3": hit_at_k(indice, MIS_PREGUNTAS, 3)}
print(informe_eval)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert len(MIS_PREGUNTAS) >= 8, "necesitas al menos 8 preguntas de evaluación"
    assert all(e in set(corpus["id"]) for _, e in MIS_PREGUNTAS), "cada id esperado debe existir en tu corpus"
    assert informe_eval["n_preguntas"] == len(MIS_PREGUNTAS)
    assert informe_eval["hit@1"] == hit_at_k(indice, MIS_PREGUNTAS, 1)
    assert informe_eval["hit@3"] == hit_at_k(indice, MIS_PREGUNTAS, 3)
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'informe_eval = {"n_preguntas": len(MIS_PREGUNTAS), "hit@1": hit_at_k(indice, MIS_PREGUNTAS, 1), "hit@3": hit_at_k(indice, MIS_PREGUNTAS, 3)}.')
    print("   Detalle:", e)

## 4. Responde con citas… o abstente

El corazón del asistente (R3-03): prompt con los trozos **citados**, instrucción de usar **solo** esos documentos, y **dos barreras** de abstención — el umbral de recuperación y la salida `NO_ENCONTRADO` del LLM. Con n-gramas de caracteres los puntajes son más altos que con palabras, por eso aquí el umbral es **0,30** (en la profundización lo calibras con evidencia).

Completa las dos barreras. Recuerda: `llm` es un parámetro para poder probar con LLM «falsos».

### ✍️ Ejercicio 4 — Completa las barreras de `responder`

In [ ]:
UMBRAL = 0.30
NO_SE = ("No encontré esa información en los documentos disponibles. "
         "Te sugiero consultar con una persona de atención.")

def armar_prompt(pregunta, contexto):
    bloques = [f"[Doc {r.id} · {r.url}]\n{r.texto}" for r in contexto.itertuples()]
    return ("Eres un asistente de atención del Estado de Chile. Responde en lenguaje claro usando SOLO "
            "los documentos. Cita entre corchetes el documento usado, por ejemplo [Doc 9715]. "
            "Si los documentos no contienen la respuesta, responde exactamente: NO_ENCONTRADO\n\n"
            "Documentos:\n" + "\n\n".join(bloques) + f"\n\nPregunta: {pregunta}\nRespuesta:")

def responder(pregunta, k=3, llm=preguntar_llm):
    contexto = recuperar(indice, pregunta, k)
    if contexto["puntaje"].iloc[0] < UMBRAL:                      # barrera 1
        return {"respuesta": NO_SE, "fuentes": []}
    primero = contexto.iloc[0]
    respaldo = (f"(Modo sin LLM) Revisa «{primero['titulo']}» [Doc {primero['id']}]: "
                + " ".join(primero["texto"].split()[:50]) + "…")
    texto = llm(armar_prompt(pregunta, contexto), respaldo=respaldo)
    if "NO_ENCONTRADO" in texto:                                   # barrera 2
        return {"respuesta": NO_SE, "fuentes": []}
    return {"respuesta": texto, "fuentes": list(contexto["url"].unique())}

r = responder("¿Cómo pido la Beca Bicentenario?")
print(r["respuesta"][:400]); print("Fuentes:", r["fuentes"])

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    def llm_falso(prompt, respaldo=""):
        return "Postula en el FUAS [Doc 2227]."
    def llm_no_sabe(prompt, respaldo=""):
        return "NO_ENCONTRADO"
    def llm_prohibido(prompt, respaldo=""):
        raise RuntimeError("¡no se debía llamar al LLM!")
    url_esperada = corpus.loc[corpus["id"] == MIS_PREGUNTAS[7][1], "url"].iloc[0]
    dentro = responder(MIS_PREGUNTAS[7][0], llm=llm_falso)
    assert dentro["respuesta"] != NO_SE and url_esperada in dentro["fuentes"], "debe responder y citar el documento esperado"
    assert responder(MIS_PREGUNTAS[7][0], llm=llm_no_sabe) == {"respuesta": NO_SE, "fuentes": []}, "barrera 2"
    assert responder("¿Quién ganó la Copa América 2015?", llm=llm_prohibido) == {"respuesta": NO_SE, "fuentes": []}, "barrera 1, sin llamar al LLM"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Barrera 1: if contexto["puntaje"].iloc[0] < UMBRAL: return {"respuesta": NO_SE, "fuentes": []}. Barrera 2: if "NO_ENCONTRADO" in texto: return lo mismo.')
    print("   Detalle:", e)

## 5. Tu demo

Prueba tu asistente con preguntas de tu público — incluidas algunas que **no** debería saber
responder. Esta salida (con y sin clave) es parte de tu entrega.

In [ ]:
DEMO = ["¿Cómo pido la Beca Bicentenario?",
        "¿Hay ayuda para pagar la casa de reposo de mi mamá?",
        "¿Quién ganó la Copa América 2015?"]
try:
    for p in DEMO:
        r = responder(p)
        print(f"▶ {p}\n  {r['respuesta'][:300]}\n  fuentes: {r['fuentes']}\n")
except Exception as e:
    print("Completa los ejercicios anteriores para ver la demo.", e)

## 6. Ficha de uso responsable (riesgos y ética)

Toda solución de IA en el Estado se entrega con esta ficha. Complétala para **tu** proyecto:

> **Uso previsto y público:** _(qué preguntas responde, para quién, por qué canal)_
>
> **Fuentes y vigencia:** _(qué documentos usa, quién los mantiene, cada cuánto se actualizan; una
> ficha desactualizada produce respuestas «bien citadas» pero equivocadas)_
>
> **Evaluación:** _(n.º de preguntas, hit@1, hit@3, y cómo se comporta con preguntas fuera del corpus)_
>
> **Privacidad:** _(qué datos personales podrían llegar en las preguntas; cómo se evita enviarlos al
> LLM externo y guardarlos en registros)_
>
> **Sesgos y exclusión:** _(¿a quién le funciona peor? lenguaje coloquial, pueblos originarios,
> baja alfabetización digital, personas mayores)_
>
> **Supervisión humana:** _(quién revisa, en qué casos se deriva a una persona, cómo se corrige un error)_
>
> **Costo:** _(capa gratuita, tokens estimados por consulta, qué pasa si se cae el servicio del LLM)_
>
> **Lo que el asistente NO hace:** _(no decide beneficios, no reemplaza la atención, no interpreta casos particulares)_

## Rúbrica de evaluación

| Criterio | Qué se evalúa |
|---|---|
| **Utilidad pública** | Resuelve consultas reales que hoy consumen tiempo de atención. |
| **Evaluación con evidencia** | Set de ≥ 8 preguntas escrito antes de mirar resultados; hit@1 y hit@3 reportados, con sus límites. |
| **Groundedness / citas** | Cada respuesta cita el documento del que sale; sin fuente, no se responde. |
| **Abstención** | Se abstiene ante preguntas fuera del corpus (umbral + `NO_ENCONTRADO`) y deriva a una persona. |
| **Riesgos y ética** | Ficha de uso responsable completa: privacidad, sesgos, supervisión humana, vigencia de las fuentes. |
| **Reproducibilidad y costo** | El notebook corre de punta a punta **sin clave ni red**; costo del LLM estimado. |
| **Comunicación** | Demo clara y ficha comprensible para una jefatura no técnica. |
| **Pertinencia** | (Bonus) El corpus o los datos son de tu propio organismo o responden una pregunta real de gestión. |

> **Entregable:** notebook reproducible + demo + informe de evaluación + ficha de uso responsable.